In [2]:
from pathlib import Path
import sys
import pandas as pd

# Xác định project root từ thư mục làm việc hiện tại
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "dataset" / "chunk" / "chunks.jsonl").is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset từ {PROJECT_ROOT}")

# Cho phép import các module trong project
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_PATH = PROJECT_ROOT / "dataset" / "chunk" / "chunks.jsonl"
BM25_PATH = PROJECT_ROOT / "dataset" / "index" / "parse"
DENSE_PATH = PROJECT_ROOT / "dataset" / "index" / "dense"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("BM25 index:", BM25_PATH)
print("Dense index:", DENSE_PATH)

Project root: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT
Dataset: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\chunk\chunks.jsonl
BM25 index: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\index\parse
Dense index: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\dataset\index\dense


In [3]:
from dataset.index.source.fusion import RRFFusion

# Tạo hai danh sách kết quả giả định
bm25_results = [
    {"chunk_id": "A", "rank": 1, "retriever": "bm25"},
    {"chunk_id": "B", "rank": 2, "retriever": "bm25"},
    {"chunk_id": "C", "rank": 3, "retriever": "bm25"}
]
dense_results = [
    {"chunk_id": "C", "rank": 1, "retriever": "dense"},
    {"chunk_id": "A", "rank": 2, "retriever": "dense"},
    {"chunk_id": "D", "rank": 3, "retriever": "dense"}
]

# Tính RRF score với k=60
fusion = RRFFusion(k=60)
results = fusion.fuse([bm25_results, dense_results], top_k=4)

for result in results:
    print(result)

# Kiểm tra thứ hạng theo kết quả tính tay
assert [r["chunk_id"] for r in results] == ["A", "C", "B", "D"]
assert abs(results[0]["rrf_score"] - (1/61 + 1/62)) < 1e-10

print("RRF unit test passed.")


{'chunk_id': 'A', 'rrf_score': 0.03252247488101534, 'rank': 1, 'retrieval_sources': ['bm25', 'dense']}
{'chunk_id': 'C', 'rrf_score': 0.032266458495966696, 'rank': 2, 'retrieval_sources': ['bm25', 'dense']}
{'chunk_id': 'B', 'rrf_score': 0.016129032258064516, 'rank': 3, 'retrieval_sources': ['bm25']}
{'chunk_id': 'D', 'rrf_score': 0.015873015873015872, 'rank': 4, 'retrieval_sources': ['dense']}
RRF unit test passed.


TEST THỰC TẾ

In [4]:
from dataset.index.source.common import load_chunks

# Đọc toàn bộ chunks đã được đánh dấu index_eligible
chunks = load_chunks(DATASET_PATH)

# Tạo dictionary ánh xạ chunk_id -> original record
chunk_store = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

print("Total chunks:", len(chunks))
print("Total records in chunk_store:", len(chunk_store))
print("First chunk ID:", chunks[0]["chunk_id"])

Total chunks: 621
Total records in chunk_store: 621
First chunk ID: f226bf39cf7c2725961dc116a550ed2a28f9b06c5a9652cf03b1a27ea71c117e


In [5]:
from dataset.index.source.sparse import SparseRetriever
from dataset.index.source.dense import DenseRetriever

# Load BM25 đã xây dựng trước đó
sparse = SparseRetriever()
sparse.load(BM25_PATH)

# Load FAISS index và ánh xạ chunk ID
dense = DenseRetriever()
dense.load(DENSE_PATH)

# Kiểm tra số lượng vector, documents và ID
print("BM25 documents:", sparse.bm25.corpus_size)
print("FAISS vectors:", dense.index.ntotal)
print("Embedding dimension:", dense.index.d)

assert len(sparse.chunk_ids) == len(chunks)
assert len(dense.chunk_ids) == len(chunks)
assert set(sparse.chunk_ids) == set(chunk_store)
assert set(dense.chunk_ids) == set(chunk_store)

print("Đã load thành công cả hai index.")

c:\Users\Nguyen Tri Nhan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BM25 documents: 621
FAISS vectors: 621
Embedding dimension: 1024
Đã load thành công cả hai index.


In [6]:
from dataset.index.source.hybrid import HybridRetriever

# Khởi tạo hệ thống Hybrid Retrieval
hybrid = HybridRetriever(
    sparse=sparse,
    dense=dense,
    rrf_k=60,
    retriever_top_k=20,
    fusion_top_k=7
)

# Thực hiện retrieval
query = "Tiêu chuẩn chẩn đoán đái tháo đường típ 2 là gì?"
output = hybrid.search(query)

# Hiển thị số lượng kết quả
print("BM25:", len(output["sparse_results"]))
print("BGE-M3:", len(output["dense_results"]))
print("RRF:", len(output["fused_results"]))

# Lookup chunk gốc và hiển thị nội dung
for result in output["fused_results"]:
    chunk_id = result["chunk_id"]
    original = chunk_store.get(chunk_id)

    print(f"\nRank: {result['rank']}")
    print(f"RRF Score: {result['rrf_score']:.6f}")
    print(f"Retrievers: {result['retrieval_sources']}")
    print(f"Chunk ID: {chunk_id}")
    print(f"Document: {original['document_id'] if original else 'NOT FOUND'}")
    print(f"Text: {original['text'][:500] if original else 'NOT FOUND'}")
    print("-" * 70)


Loading BGE-M3 on cuda:0


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 291.48it/s]


BM25: 20
BGE-M3: 20
RRF: 7

Rank: 1
RRF Score: 0.032522
Retrievers: ['bm25', 'dense']
Chunk ID: 6fb76789a5f3fbe8ce27d86226bdab604751ebe89a78edcbedb14db8e15904c0
Document: vn_moh_guideline_t2dm_2020
Text: 1.1. Chẩn đoán đái tháo đường Tiêu chuẩn chẩn đoán đái tháo đường dựa vào 1 trong 4 tiêu chí sau đây: a) Glucose huyết tương lúc đói ≥ 126 mg/dL (hay 7 mmol/L) hoặc: b) Glucose huyết tương ở thời điểm sau 2 giờ làm nghiệm pháp dung nạp với 75g glucose bằng đường uống ≥ 200 mg/dL (hay 11,1 mmol/L) c) HbA1c ≥ 6,5% (48 mmol/mol). Xét nghiệm HbA1c phải được thực hiện bằng phương pháp đã chuẩn hóa theo tiêu chuẩn quốc tế. d) BN có triệu chứng kinh điển của tăng glucose huyết hoặc của cơn tăng glucose
----------------------------------------------------------------------

Rank: 2
RRF Score: 0.030478
Retrievers: ['bm25', 'dense']
Chunk ID: 9f7ad91b174d0f835a215881827acca5cfedfbcb3b9d404d8683dcfe5c23dcaf
Document: vn_moh_guideline_t2dm_2020
Text: 2. Phân loại đái tháo đường Bệnh đái tháo đường

Kiểm thử

In [7]:
# Lấy tập ID của mỗi retriever
bm25_ids = {
    result["chunk_id"]
    for result in output["sparse_results"]
}
dense_ids = {
    result["chunk_id"]
    for result in output["dense_results"]
}

# Đếm số chunks được tìm thấy bởi cả hai phương pháp
overlap = bm25_ids & dense_ids

print("BM25 results:", len(bm25_ids))
print("BGE-M3 results:", len(dense_ids))
print("Overlap:", len(overlap))
print("Unique candidates:", len(bm25_ids | dense_ids))

BM25 results: 20
BGE-M3 results: 20
Overlap: 8
Unique candidates: 32


In [8]:
# Kiểm tra các chunks sau RRF có nằm trong cả hai danh sách không
overlap_ids = bm25_ids & dense_ids
fused_ids = {
    result["chunk_id"]
    for result in output["fused_results"]
}
print("Top-7 thuộc overlap:", len(fused_ids & overlap_ids))
print("Top-7 chỉ thuộc một retriever:", len(fused_ids - overlap_ids))

Top-7 thuộc overlap: 7
Top-7 chỉ thuộc một retriever: 0


In [10]:

from pathlib import Path

# Danh sách câu hỏi và tên file xuất kết quả
evaluation_queries = [
    {
        "id": "Q01",
        "query": "Người bệnh đái tháo đường típ 2 cần thực hiện những xét nghiệm nào để theo dõi và đánh giá biến chứng thận?",
        "filename": "Q01_kidney_monitoring.txt"
    },
    {
        "id": "Q02",
        "query": "Metformin được sử dụng như thế nào trong điều trị đái tháo đường típ 2 và có những lưu ý gì khi sử dụng?",
        "filename": "Q02_metformin_treatment.txt"
    },
    {
        "id": "Q03",
        "query": "What are the recommended approaches for managing cardiovascular risk in patients with type 2 diabetes?",
        "filename": "Q03_cardiovascular_management.txt"
    },
    {
        "id": "Q04",
        "query": "Những yếu tố nào làm tăng nguy cơ mắc bệnh tim mạch ở người bị tiểu đường?",
        "filename": "Q04_cardiovascular_risk.txt"
    },
    {
        "id": "Q05",
        "query": "How does insulin resistance contribute to the development of type 2 diabetes?",
        "filename": "Q05_insulin_resistance.txt"
    }
]

# Thư mục lưu kết quả kiểm thử
OUTPUT_DIR = PROJECT_ROOT / "test" / "test_results_chunks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ghi toàn bộ retrieval results, không cắt ngắn nội dung chunks
def write_retrieval_section(f, title: str, results: list[dict], chunk_store: dict):
    f.write(f"\n{'=' * 100}\n")
    f.write(f"{title} | TOTAL: {len(results)}\n")
    f.write(f"{'=' * 100}\n")
    for result in results:
        chunk_id = result["chunk_id"]
        chunk = chunk_store.get(chunk_id)
        f.write(f"\nRANK: {result['rank']}\n")
        f.write(f"CHUNK ID: {chunk_id}\n")
        if "rrf_score" in result:
            f.write(f"RRF SCORE: {result['rrf_score']:.6f}\n")
            f.write(f"RETRIEVERS: {result['retrieval_sources']}\n")
        else:
            f.write(f"SCORE: {result['score']:.6f}\n")
            f.write(f"RETRIEVER: {result['retriever']}\n")
        if chunk is None:
            f.write("STATUS: CHUNK NOT FOUND\n")
        else:
            f.write(f"DOCUMENT: {chunk.get('document_id')}\n")
            f.write(f"VERSION: {chunk.get('version_id')}\n")
            f.write(f"LANGUAGE: {chunk.get('language')}\n")
            f.write(f"SECTION: {chunk.get('section_path')}\n")
            f.write(f"SOURCE URL: {chunk.get('source_url')}\n")
            f.write("\nFULL TEXT:\n")
            f.write(chunk["text"] + "\n")
        f.write("-" * 100 + "\n")

# Chạy retrieval cho từng query và lưu mỗi query vào một file
for item in evaluation_queries:
    output = hybrid.search(item["query"], top_k=7)
    sparse_results = output["sparse_results"]
    dense_results = output["dense_results"]
    fused_results = output["fused_results"]
    bm25_ids = {r["chunk_id"] for r in sparse_results}
    dense_ids = {r["chunk_id"] for r in dense_results}
    overlap = bm25_ids & dense_ids
    file_path = OUTPUT_DIR / item["filename"]

    with file_path.open("w", encoding="utf-8") as f:
        f.write(f"QUERY ID: {item['id']}\n")
        f.write(f"QUERY: {item['query']}\n")
        f.write("CONFIG: BM25 Top-20 | BGE-M3 Top-20 | RRF k=60 | RRF Top-7\n")
        write_retrieval_section(f, "BM25 TOP-20", sparse_results, chunk_store)
        write_retrieval_section(f, "BGE-M3 TOP-20", dense_results, chunk_store)
        write_retrieval_section(f, "RRF TOP-7", fused_results, chunk_store)
        f.write(f"\n{'=' * 100}\nSUMMARY\n")
        f.write(f"BM25 RESULTS: {len(bm25_ids)}\n")
        f.write(f"BGE-M3 RESULTS: {len(dense_ids)}\n")
        f.write(f"OVERLAP: {len(overlap)}\n")
        f.write(f"UNIQUE CANDIDATES: {len(bm25_ids | dense_ids)}\n")
        f.write(f"RRF RESULTS: {len(fused_results)}\n")

    print(f"[{item['id']}] Saved: {file_path.name} | Overlap: {len(overlap)}")

print(f"\nCompleted: {len(evaluation_queries)} queries")
print(f"Output directory: {OUTPUT_DIR}")


[Q01] Saved: Q01_kidney_monitoring.txt | Overlap: 11
[Q02] Saved: Q02_metformin_treatment.txt | Overlap: 4
[Q03] Saved: Q03_cardiovascular_management.txt | Overlap: 4
[Q04] Saved: Q04_cardiovascular_risk.txt | Overlap: 4
[Q05] Saved: Q05_insulin_resistance.txt | Overlap: 6

Completed: 5 queries
Output directory: D:\1_TaiLieuHocTap\2627\DuAn_KhoaLuan\Demo_Nhan\DACNTT\test\test_results_chunks
